# Lab 3B: 세분화된 액세스 제어를 적용한 문제 해결 에이전트

## 개요
이 Lab에서는 Lab 3A를 기반으로 JWT 인증과 Lambda 인터셉터를 사용하는 역할 기반 액세스 제어를 도입합니다. SRE 사용자는 문제 해결 계획을 수립할 수 있지만 승인자만 실행할 수 있도록 직무 분리 패턴을 구현합니다.

## 학습 목표
- Cognito를 사용한 JWT 기반 인증 이해
- Lambda 인터셉터를 통한 역할 기반 액세스 제어(RBAC) 구현
- 사용자 지정 권한 부여를 적용한 AgentCore Gateway 배포 및 구성
- 자동화된 문제 해결 워크플로에서 직무 분리 적용
- 액세스 결정 모니터링 및 감사

## 사용자 역할 및 액세스 모델

### SRE 역할
- **권한:** 문제 해결 계획 생성 및 검증
- **제한 사항:** 문제 해결 작업 실행 불가
- **사용 사례:** 문제를 진단하고 해결 방법 제안

### 승인자 역할
- **권한:** 문제 해결 작업 실행 및 검증
- **제한 사항:** 없음(전체 액세스)
- **사용 사례:** SRE 계획을 검토하고 승인된 해결 작업 실행

---

# 1단계: 사용 사례 설명

## 시나리오

조직에서 인프라 변경에 대한 **직무 분리**가 필요합니다.

- **SRE 팀:** 문제를 진단하고 문제 해결 계획 수립
- **승인자 팀:** 승인된 변경 사항을 검토하고 실행

**문제:** 액세스 제어가 없으면 누구나 인프라 변경을 실행할 수 있어 규정 준수 및 보안 위험이 발생합니다.

**해결 방법:** JWT 인증과 Lambda 인터셉터를 사용하여 Gateway 수준에서 역할 기반 권한을 적용합니다.

## JWT + Lambda 인터셉터를 사용하는 이유

**JWT 인증:**
- 업계 표준 토큰 형식
- 사용자 식별 정보 및 그룹 소속 정보 포함
- Cognito가 암호학적으로 서명
- 데이터베이스 조회 불필요

**Lambda 인터셉터:**
- Gateway의 REQUEST 단계에서 실행
- Runtime으로 라우팅하기 전에 JWT 클레임 검사
- 세분화된 액세스 제어 적용
- CloudWatch 로그를 통해 감사 추적 정보 제공

**장점:**
- 권한 부여 로직 중앙 집중화
- 에이전트 코드 변경 불필요
- 확장 가능한 서버리스 방식
- 간편한 감사 및 수정

## 워크플로 다이어그램

```
┌─────────────────────────────────────────────────────────────────┐
│                    SRE Workflow (Planning)                      │
└─────────────────────────────────────────────────────────────────┘

SRE User → Cognito Auth → JWT (groups: ["sre"])
                                ↓
                          Gateway receives token
                                ↓
                    Lambda Interceptor checks:
                    - Extract cognito:groups = ["sre"]
                    - Tool: generate_remediation_plan
                    - Permission: ✅ ALLOWED
                                ↓
                          Runtime executes
                                ↓
                    Remediation plan created ✅

┌─────────────────────────────────────────────────────────────────┐
│              SRE Workflow (Execution Attempt)                   │
└─────────────────────────────────────────────────────────────────┘

SRE User → Cognito Auth → JWT (groups: ["sre"])
                                ↓
                          Gateway receives token
                                ↓
                    Lambda Interceptor checks:
                    - Extract cognito:groups = ["sre"]
                    - Tool: execute_remediation_step
                    - Permission: ❌ DENIED
                                ↓
                    Request blocked at gateway ❌

┌─────────────────────────────────────────────────────────────────┐
│                 Approver Workflow (Execution)                   │
└─────────────────────────────────────────────────────────────────┘

Approver → Cognito Auth → JWT (groups: ["approvers"])
                                ↓
                          Gateway receives token
                                ↓
                    Lambda Interceptor checks:
                    - Extract cognito:groups = ["approvers"]
                    - Tool: execute_remediation_step
                    - Permission: ✅ ALLOWED
                                ↓
                          Runtime executes
                                ↓
                    Remediation executed ✅
```

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

In [ ]:
# Lab 3A 완료 여부 확인
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS

try:
    user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])
    print("✅ Lab 3A prerequisites verified")
    print(f"   Cognito User Pool: {user_pool_id}")
    print("   Ready to proceed with Lab 3B")
except Exception:
    print("❌ Lab 3A not complete. Please run Lab-03a-remediation-agent.ipynb first.")
    raise

---

# 2단계: Cognito 토큰 및 그룹 클레임

## 목표
Cognito 토큰에 포함된 내용과 그룹 클레임을 통해 액세스 제어를 적용하는 방법을 이해합니다.

## 실습 항목
- [ ] SRE 사용자의 Cognito 토큰 가져오기
- [ ] 승인자 사용자의 Cognito 토큰 가져오기
- [ ] JWT 토큰 디코딩
- [ ] 두 토큰의 그룹 클레임 표시
- [ ] 토큰 페이로드 간 차이 비교

In [ ]:
# 모듈 가져오기
import json
import boto3
from datetime import datetime
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
from lab_helpers.lab_03 import decode_jwt, print_token_claims, compare_tokens

cognito = boto3.client("cognito-idp", region_name=AWS_REGION)

### SRE 사용자 토큰

In [ ]:
# SRE 사용자 인증
user_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
sre_username = get_parameter(PARAMETER_PATHS["cognito"]["test_user_email"])
sre_password = get_parameter(PARAMETER_PATHS["cognito"]["test_user_password"])

sre_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": sre_username, "PASSWORD": sre_password},
)

sre_token = sre_response["AuthenticationResult"]["AccessToken"]
print(f"✅ SRE authenticated: {sre_username}")
print(f"   Token (first 50 chars): {sre_token}")

In [ ]:
# SRE 토큰 디코딩 및 표시
sre_claims = decode_jwt(sre_token)
print_token_claims(sre_claims, "SRE Token Claims")

### 승인자 토큰

In [ ]:
# 승인자 인증
approver_username = get_parameter(PARAMETER_PATHS["cognito"]["approver_user_email"])
approver_password = get_parameter(PARAMETER_PATHS["cognito"]["approver_user_password"])

approver_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": approver_username, "PASSWORD": approver_password},
)

approver_token = approver_response["AuthenticationResult"]["AccessToken"]
print(f"✅ Approver authenticated: {approver_username}")
print(f"   Token (first 50 chars): {approver_token[:50]}...")

In [ ]:
# 승인자 토큰 디코딩 및 표시
approver_claims = decode_jwt(approver_token)
print_token_claims(approver_claims, "Approver Token Claims")

### 토큰 페이로드 비교

In [ ]:
# 토큰을 나란히 비교
compare_tokens(sre_claims, approver_claims)

### 핵심 개념

**JWT 구조:**
- `header.payload.signature`
- 페이로드에는 클레임(사용자 속성)이 포함됨
- 서명은 토큰의 진위 여부를 검증함

**Cognito의 그룹 클레임:**
- 토큰에 `cognito:groups`가 자동으로 포함됨
- 사용자가 속한 모든 그룹이 나열됨
- 사용자 지정 속성이 필요하지 않음

**권한 부여 흐름:**
```
User → Cognito → JWT with cognito:groups
                        ↓
                  Gateway receives token
                        ↓
                Lambda Interceptor extracts cognito:groups
                        ↓
                Maps groups to allowed tools
                        ↓
                Allow/Deny decision
```

**권한 매핑(Lambda에서 구현 예정):**
```python
GROUP_PERMISSIONS = {
    "sre": ["generate_remediation_plan"],
    "approvers": ["execute_remediation_step", "validate_remediation_environment"]
}
```

---

# 3단계: 인터셉터 Lambda 배포

## 목표
Gateway 수준에서 액세스 제어를 적용할 Lambda 함수를 생성하고 배포합니다.

## 인터셉터 로직
Lambda는 수신 요청을 검사하여 다음 작업을 수행합니다.
- [ ] 요청 헤더에서 JWT 추출
- [ ] JWT 서명 검증
- [ ] 그룹 클레임 확인
- [ ] 작업 기반 액세스 제어 적용
- [ ] 허용/거부 결정 반환

### 인터셉터 코드 검토

In [ ]:
# 인터셉터 Lambda 코드 검토
with open("lab_helpers/lab_03/interceptor-request.py", "r") as f:
    print(f.read())

### Lambda 함수 배포

In [ ]:
# us-west-2에 인터셉터 Lambda 배포
from lab_helpers.lab_03 import deploy_interceptor
from lab_helpers.parameter_store import put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION, WORKSHOP_NAME

print("📦 Deploying interceptor Lambda...")
function_arn = deploy_interceptor(region=AWS_REGION, prefix=WORKSHOP_NAME)

# 이후 사용을 위해 저장
put_parameter(PARAMETER_PATHS["lab_03b"]["interceptor_function_arn"], function_arn)

print(f"\n✅ Interceptor ready: {function_arn}")

**참고:** 8~10단계에서는 서로 다른 사용자 토큰으로 Gateway를 호출하여 인터셉터를 테스트합니다.

---

# 4단계: Lab 3A Gateway 정리

## 목표
Lab 3A의 Gateway에는 세분화된 액세스 제어에 필요한 Lambda 인터셉터 구성이 없으므로 해당 Gateway를 삭제합니다.

## Gateway를 삭제하는 이유

**Lab 3A Gateway 구성:**
- 인증이 필요하지 않음
- 연결된 인터셉터가 없음
- 모든 도구에 대한 액세스가 허용됨

**Lab 3B Gateway 요구 사항:**
- Cognito를 통한 JWT 인증
- REQUEST 단계의 Lambda 인터셉터
- 역할 기반 액세스 제어 적용

**업데이트할 수 없는 이유**
Gateway 인터셉터 구성은 생성 후 수정할 수 없으며 최초 배포 시 설정해야 합니다. 따라서 다음 작업이 필요합니다.
1. Lab 3A Gateway 삭제
2. 인터셉터 구성이 적용된 새 Gateway 생성
3. 기존 Runtime 재사용(변경 불필요)

**삭제되는 리소스:**
- ✅ Gateway(인터셉터 구성 필요)
- ✅ Gateway Target(Gateway와 함께 자동으로 삭제됨)

**재사용되는 리소스:**
- ✅ Runtime(동일한 에이전트 로직, 다른 Gateway)
- ✅ Cognito User Pool(이미 구성됨)
- ✅ Lambda 인터셉터(3단계에서 배포됨)

### Lab 3A Gateway 삭제

In [ ]:
import boto3
import time

# bedrock-agentcore-control 클라이언트 초기화
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# 모든 Gateway 나열
print("🔍 Listing all gateways...")
response = agentcore_client.list_gateways()
gateways = response.get("items", [])

if not gateways:
    print("ℹ️  No gateways found")
else:
    print(f"📋 Found {len(gateways)} gateway(s)\n")

    # Lab 3B로 교체할 Lab 3A Gateway만 필터링
    lab_3a_gateways = [gw for gw in gateways if "aiml301-remediation-gateway" in gw.get("name", "")]

    if not lab_3a_gateways:
        print("ℹ️  No Lab 3A gateways to delete")
    else:
        print(f"🗑️  Deleting {len(lab_3a_gateways)} Lab 3A gateway(s)\n")

        # 각 Lab 3A Gateway 삭제
        for gateway in lab_3a_gateways:
            gateway_id = gateway["gatewayId"]
            gateway_name = gateway.get("name", "N/A")

            print(f"🗑️  Deleting: {gateway_name} ({gateway_id})")

            try:
                # 먼저 모든 target을 나열하고 삭제
                targets_response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
                targets = targets_response.get("items", [])

                for target in targets:
                    target_id = target["targetId"]
                    print(f"   🎯 Deleting target: {target_id}")
                    agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)

                # 모든 target이 삭제될 때까지 대기
                if targets:
                    print("   ⏳ Waiting for targets to be deleted...")
                    for _ in range(30):
                        time.sleep(2)
                        check = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
                        if len(check.get("items", [])) == 0:
                            break

                # Gateway 삭제
                agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
                print("   ✅ Deleted")

            except Exception as e:
                print(f"   ❌ Error: {e}")

        print("\n✅ Cleanup complete")

**참고:** Runtime은 Lab 3A의 리소스를 재사용하므로 삭제할 필요가 없습니다.

---

# 5단계: JWT 인증을 적용한 새 Gateway 및 Target 생성

## 목표
JWT 인증과 Lambda 인터셉터가 구성된 새 Gateway를 배포합니다.

### Gateway 구성

**Gateway 세부 정보:**
- 이름: `interceptor-gateway-jwt-[random]`
- 프로토콜: MCP
- 권한 부여자 유형: CUSTOM_JWT
- 인터셉트 지점: REQUEST

**JWT 구성:**
```json
{
  "authorizerType": "CUSTOM_JWT",
  "discoveryUrl": "https://cognito-idp.us-west-2.amazonaws.com/us-west-2_POOL_ID/.well-known/openid-configuration",
  "allowedClients": [
    "CLIENT_ID_1",
    "CLIENT_ID_2"
  ]
}
```

**인터셉터 구성:**
```json
{
  "interceptionPoints": ["REQUEST"],
  "interceptor": {
    "lambda": {
      "arn": "arn:aws:lambda:us-west-2:ACCOUNT:function:custom-interceptor-request"
    }
  },
  "inputConfiguration": {
    "passRequestHeaders": true
  }
}
```


### Gateway 생성

In [ ]:
import random
import string
import time
import boto3
from lab_helpers.parameter_store import get_parameter, put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
from lab_helpers.lab_03.gateway_setup import AgentCoreGatewaySetup

# bedrock-agentcore-control 클라이언트 초기화
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# 구성 값 가져오기
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])
user_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
m2m_client_id = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_id"])
interceptor_arn = get_parameter(PARAMETER_PATHS["lab_03b"]["interceptor_function_arn"])

# us-west-2에 Gateway IAM 역할 생성
gateway_setup = AgentCoreGatewaySetup(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=False)
role_info = gateway_setup.create_gateway_service_role()
role_arn = role_info["role_arn"]
print(f"✅ Gateway role: {role_arn}")

# 고유한 Gateway 이름 생성
suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=10))
gateway_name = f"aiml301-interceptor-gateway-jwt-{suffix}"

print(f"📤 Creating gateway: {gateway_name}")
print("   Gateway region: us-west-2")
print(f"   Cognito region: {AWS_REGION}")
print(f"   Role: {role_arn}")
print(f"   Interceptor: {interceptor_arn}")

# Gateway 생성
response = agentcore_client.create_gateway(
    name=gateway_name,
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration",
            "allowedClients": [user_client_id, m2m_client_id],
        }
    },
    interceptorConfigurations=[
        {
            "interceptionPoints": ["REQUEST"],
            "interceptor": {"lambda": {"arn": interceptor_arn}},
            "inputConfiguration": {"passRequestHeaders": True},
        }
    ],
    roleArn=role_arn,
)

gateway_id = response["gatewayId"]
gateway_arn = response["gatewayArn"]

print(f"\n✅ Gateway created: {gateway_id}")
print(f"   ARN: {gateway_arn}")

In [ ]:
# Gateway가 READY 상태가 될 때까지 대기
print("⏳ Waiting for gateway to be ready...")
while True:
    response = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)

    status = response["status"]
    if status == "READY":
        gateway_url = response["gatewayUrl"]
        print(f"✅ Gateway ready: {status}")
        print(f"   URL: {gateway_url}")
        break
    elif status == "FAILED":
        print("❌ Gateway creation failed")
        break
    print(f"   Status: {status}")
    time.sleep(5)

# 이후 사용을 위해 저장
put_parameter(PARAMETER_PATHS["lab_03b"]["gateway_id"], gateway_id)
put_parameter(PARAMETER_PATHS["lab_03b"]["gateway_url"], gateway_url)

### Gateway 구성 확인

In [ ]:
# 전체 Gateway 구성 표시
result = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)

print("📋 Gateway Configuration:")
print("=" * 70)
print(json.dumps(result, indent=2, default=str))
print("=" * 70)

# 주요 구성 요소 강조
print("\n🔑 Key Configuration Elements:")
print(f"   Gateway ID: {result['gatewayId']}")
print(f"   Gateway URL: {result['gatewayUrl']}")
print(f"   Protocol: {result['protocolType']}")
print(f"   Authorizer: {result['authorizerType']}")

# JWT 구성
jwt_config = result["authorizerConfiguration"]["customJWTAuthorizer"]
print("\n🔐 JWT Configuration:")
print(f"   Discovery URL: {jwt_config['discoveryUrl']}")
print(f"   Allowed Clients: {len(jwt_config['allowedClients'])} clients")
for i, client in enumerate(jwt_config["allowedClients"], 1):
    print(f"      {i}. {client}")

# 인터셉터 구성
interceptor_config = result["interceptorConfigurations"][0]
print("\n🛡️  Interceptor Configuration:")
print(f"   Lambda ARN: {interceptor_config['interceptor']['lambda']['arn']}")
print(f"   Interception Points: {interceptor_config['interceptionPoints']}")
print(f"   Pass Request Headers: {interceptor_config['inputConfiguration']['passRequestHeaders']}")
print("\n   ℹ️  The interceptor will:")
print("      • Examine JWT tokens in Authorization header")
print("      • Extract cognito:groups claim")
print("      • Enforce group-based permissions")
print("      • Allow/deny requests before reaching runtime")

인터셉터가 구성된 Gateway에서 Lambda를 호출할 수 있도록 권한을 추가합니다.

In [ ]:
# Gateway ARN으로 Lambda 권한 업데이트
import boto3

lambda_client = boto3.client("lambda", region_name=AWS_REGION)
function_name = f"{WORKSHOP_NAME}-interceptor-request"
gateway_arn = result["gatewayArn"]

print("🔧 Updating Lambda permission with gateway ARN...")

# 기존 권한 제거
try:
    lambda_client.remove_permission(FunctionName=function_name, StatementId="AllowGatewayInvoke")
    print("   Removed old permission")
except:
    pass

# 소스 ARN을 지정하여 새 권한 추가
lambda_client.add_permission(
    FunctionName=function_name,
    StatementId="AllowGatewayInvoke",
    Action="lambda:InvokeFunction",
    Principal="bedrock-agentcore.amazonaws.com",
    SourceArn=gateway_arn,
)

print("✅ Lambda permission updated with source ARN")
print(f"   Gateway ARN: {gateway_arn}")

### Target 구성

**Target 세부 정보:**
- 이름: `mcp-remediation-target`
- 유형: MCP 서버
- 엔드포인트: [6단계의 Runtime 엔드포인트]

**자격 증명 공급자:**
- 유형: OAUTH
- 공급자: Cognito MCP Provider

### Target 생성

In [ ]:
# Lab 3A에서 Runtime 엔드포인트 가져오기
runtime_arn = get_parameter(PARAMETER_PATHS["lab_03"]["runtime_arn"])
runtime_endpoint = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{runtime_arn.replace(':', '%3A').replace('/', '%2F')}/invocations?qualifier=DEFAULT"
print(f"runtime endpoint: {runtime_endpoint}")
# OAuth 공급자 ARN 가져오기
oauth_provider_arn = get_parameter(PARAMETER_PATHS["lab_03"]["oauth2_provider_arn"])

# Target 생성
response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="mcp-remediation-target",
    description="MCP server target for remediation agent with JWT auth",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": runtime_endpoint}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth_provider_arn,
                    "scopes": [],
                }
            },
        }
    ],
)

target_id = response["targetId"]
print(f"✅ Target Creating: {target_id}")
print(f"   Runtime: {runtime_arn}")

In [ ]:
# Target이 READY 상태가 될 때까지 대기
print("⏳ Waiting for target to be ready...")

target_info = agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
time.sleep(5)
status = target_info.get("status", "UNKNOWN")

if status == "READY":
    print("✅ Target is READY")
    print("\n🎉 Gateway and target ready for testing!")
    print(f"   Gateway URL: {gateway_url}")
    print(f"   Target ID: {target_id}")
if status == "FAILED" or status == "SYNCHRONIZE_UNSUCCESSFUL":
    print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")

### Target 구성 확인

In [ ]:
# 전체 Target 구성 표시
result = agentcore_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)

print("📋 Target Configuration:")
print("=" * 70)
print(json.dumps(result, indent=2, default=str))
print("=" * 70)

# 주요 구성 요소 강조
print("\n🔑 Key Configuration Elements:")
print(f"   Target ID: {result['targetId']}")
print(f"   Target Name: {result['name']}")
print(f"   Status: {result['status']}")

# MCP 서버 구성
mcp_config = result["targetConfiguration"]["mcp"]["mcpServer"]
print("\n🔌 MCP Server Configuration:")
print(f"   Endpoint: {mcp_config['endpoint'][:80]}...")
print("   ℹ️  Points to Lab 3A runtime (reused)")

# 자격 증명 공급자 구성
cred_config = result["credentialProviderConfigurations"][0]
oauth_config = cred_config["credentialProvider"]["oauthCredentialProvider"]
print("\n🔐 Credential Provider Configuration:")
print(f"   Type: {cred_config['credentialProviderType']}")
print(f"   Provider ARN: {oauth_config['providerArn']}")
print("   ℹ️  Uses Cognito OAuth provider for machine-to-machine auth")

# 요청 흐름 설명
print("\n📊 Request Flow:")
print("   1. User sends request with JWT token → Gateway")
print("   2. Gateway validates JWT (Cognito OIDC)")
print("   3. Lambda interceptor checks permissions")
print("   4. If allowed → Gateway forwards to Target")
print("   5. Target uses OAuth to authenticate with Runtime")
print("   6. Runtime executes agent logic")
print("   7. Response flows back through Gateway")

---

# 6단계: MCP Runtime 검토

## 목표
Gateway를 통해 액세스할 문제 해결 에이전트 Runtime을 검토합니다.

### Runtime 구성 확인

In [ ]:
# Lab 3A의 Runtime을 사용할 수 있는지 확인
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
import boto3

runtime_arn = get_parameter(PARAMETER_PATHS["lab_03"]["runtime_arn"])
runtime_id = runtime_arn.split("/")[-1]

print("✅ Runtime from Lab 3A:")
print(f"   ARN: {runtime_arn}")
print(f"   ID: {runtime_id}")
print(f"   Region: {AWS_REGION}")

# 제어 영역 클라이언트로 Runtime 상태 확인
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
try:
    runtime_info = agentcore_control.get_agent_runtime(agentRuntimeId=runtime_id)
    status = runtime_info["status"]
    print(f"\n📊 Runtime Status: {status}")

    if status == "READY":
        print("   ✅ Runtime is ready for gateway integration")
    elif status == "FAILED" or status == "SYNCHRONIZE_UNSUCCESSFUL":
        print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")
except Exception as e:
    print(f"   ❌ Error checking runtime: {e}")

---

# 7단계: Cognito 토큰 재생성

## 목표
새 Gateway에서 유효한 두 역할의 최신 JWT 토큰을 가져옵니다.

### SRE 사용자 토큰

In [ ]:
# 2단계의 토큰이 만료되었을 경우를 대비해 SRE 토큰 재생성
sre_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": sre_username, "PASSWORD": sre_password},
)

sre_token = sre_response["AuthenticationResult"]["AccessToken"]
sre_claims = decode_jwt(sre_token)

print("✅ Fresh SRE token generated")
print(f"   Username: {sre_username}")
print(f"   Groups: {sre_claims.get('cognito:groups', [])}")
print(f"   Token (first 50 chars): {sre_token[:50]}...")

### 승인자 토큰

In [ ]:
# 2단계의 토큰이 만료되었을 경우를 대비해 승인자 토큰 재생성
approver_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": approver_username, "PASSWORD": approver_password},
)

approver_token = approver_response["AuthenticationResult"]["AccessToken"]
approver_claims = decode_jwt(approver_token)

print("✅ Fresh Approver token generated")
print(f"   Username: {approver_username}")
print(f"   Groups: {approver_claims.get('cognito:groups', [])}")
print(f"   Token (first 50 chars): {approver_token[:50]}...")

## Memory의 진단 정보로 컨텍스트 보강

선별된 Memory에서 추가 정보를 가져와 진단 정보로 컨텍스트를 보강합니다.

In [ ]:
agent_memory_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

memory_id = get_parameter(PARAMETER_PATHS["memory"]["memory_id"])
memory_session_id = get_parameter(PARAMETER_PATHS["memory"]["default_session_id"])

print(memory_id)
print(memory_session_id)
actor_id = "diagnostics_agent"


# 성공적으로 기록되었는지 확인하기 위해 AgentCore Memory에 추가된 이벤트 나열
params = {
    "memoryId": memory_id,
    "actorId": actor_id,
    "sessionId": memory_session_id,
    "includePayloads": True,
}
# 모든 메시지 가져오기
response = agent_memory_client.list_events(**params)
additional_context = ""
for event in response.get("events", []):
    payload = event.get("payload", [])
    for i, item in enumerate(payload):
        if "conversational" in item:
            text = item["conversational"]["content"]["text"]
            additional_context += text
additional_context

---

# 8단계: SRE의 계획 생성 시뮬레이션

## 목표
SRE 사용자가 문제 해결 계획을 성공적으로 생성할 수 있음을 확인합니다.

## 시나리오
SRE가 문제를 감지하고 이를 해결하기 위한 계획을 생성합니다.

### 계획 생성 실행

In [ ]:
# SRE 토큰으로 Gateway에 연결
from lab_helpers.lab_03.mcp_client import MCPClient

gateway_url = get_parameter(PARAMETER_PATHS["lab_03b"]["gateway_url"])

print("🔗 Connecting to gateway as SRE user...")
print(f"   Gateway: {gateway_url}")
print(f"   User: {sre_username}")
print(f"   Groups: {sre_claims.get('cognito:groups', [])}")

sre_client = MCPClient(gateway_url, sre_token)
sre_client.initialize()

print("\n✅ Connected to gateway")

In [ ]:
# 사용 가능한 도구 나열
print("🔧 Listing available tools...\n")
tools = sre_client.list_tools()

print(f"📋 Available tools ({len(tools)}):")
for tool in tools:
    print(f"   • {tool['name']}")
    if "description" in tool:
        print(tool)
        desc = tool["description"]
        print(f"     {desc[:80]}..." if len(desc) > 80 else f"     {desc}")

In [ ]:
# 문제 해결 계획 도구 호출
start_time = datetime.now()
print("\n🎯 Invoking generate_remediation_plan tool...\n")
try:
    result = sre_client.call_tool(
        tool_name="mcp-remediation-target___infrastructure_agent",
        arguments={
            "remediation_query": f"""I need help with infrastructure remediation for our CRM application. We're experiencing: {additional_context} """,
            "action_type": "only_plan",
        },
    )
except Exception as e:
    print(f"❌ Error: {e}")
analysis_time = (datetime.now() - start_time).total_seconds()
print(f"Analysis Time: {analysis_time:.2f} seconds")
print("✅ Tool invocation successful!")
print("\n📋 Remediation Plan:")
print(f"{result[:500]}..." if len(result) > 500 else result)

### 성공 여부 확인

In [ ]:
print("\n✅ SRE User Successfully:")
print("   1. Connected to gateway with JWT token")
print("   2. Listed available tools")
print("   3. Invoked generate_remediation_plan tool")
print("   4. Received remediation plan")
print("\n🔒 Interceptor allowed this operation because:")
print("   • User is in 'sre' group")
print("   • Tool 'generate_remediation_plan' is allowed for SRE users")

### 예상 결과
✅ SRE 토큰으로 계획이 성공적으로 생성됨

---

# 9단계: SRE의 실행 시도(액세스 거부)

## 목표
인터셉터가 차단하므로 SRE 사용자는 문제 해결 작업을 실행할 수 없음을 확인합니다.

## 시나리오
SRE가 자신이 생성한 계획의 실행을 시도합니다.

### 실행 시도

In [ ]:
# 문제 해결 계획 도구 호출
print("\n🎯 Invoking execute_remediation_step tool...\n")

result = sre_client.call_tool(
    tool_name="mcp-remediation-target___infrastructure_agent",
    arguments={
        "remediation_query": f"""
            Help me change the read and write capacity to on-demand for the DynamoDb tables in my CRM application, based on this diagnostic information: {additional_context}
            """,
        "action_type": "only_execute",
    },
)

### 액세스 거부 분석

In [ ]:
# Lambda 로그에서 인터셉터의 거부 내역 확인
# 로그가 CloudWatch에 도착하는 데 몇 분 정도 걸릴 수 있으므로 기다린 후 다시 시도
import boto3
from datetime import datetime

logs = boto3.client("logs", region_name=AWS_REGION)
log_group = "/aws/lambda/aiml301_sre_agentcore-interceptor-request"

streams = logs.describe_log_streams(logGroupName=log_group, orderBy="LastEventTime", descending=True, limit=1)

print("📋 Interceptor Lambda Logs (Last 20 lines):\n")
stream = streams["logStreams"][0]
events = logs.get_log_events(
    logGroupName=log_group,
    logStreamName=stream["logStreamName"],
    limit=30,
    startFromHead=False,
)

for event in events["events"][-20:]:
    msg = event["message"].strip()
    if any(x in msg for x in ["Tool call", "User groups", "not authorized", "Denying"]):
        print(msg)

print("\n✅ The interceptor blocked the execute_remediation_step call for SRE user")

### 예상 결과
❌ 액세스 거부 - 인터셉터의 거부 내역이 표시됨

---

# 10단계: 승인자의 실행 및 검증

## 목표
승인자 사용자가 문제 해결 작업을 실행하고 검증할 수 있음을 확인합니다.

### 파트 A: 승인자의 문제 해결 작업 실행

In [ ]:
approver_client = MCPClient(gateway_url, approver_token)
approver_client.initialize()

In [ ]:
# 문제 해결 계획 도구 호출
print("\n🎯 Invoking execute_remediation_step tool...\n")
start_time = time.time()
try:
    result = approver_client.call_tool(
        tool_name="mcp-remediation-target___infrastructure_agent",
        arguments={
            "remediation_query": """
            This is a test invocation to validate access to infrastructure_agent and tool. Please do a health check and confirm your status.
            """,
            "action_type": "only_execute",
        },
    )
except Exception as e:
    print(f"❌ Error: {e}")
end_time = time.time()
print(f"  ⏰ Total time taken: {end_time - start_time:.2f} seconds")

### 파트 B: 승인자의 해결 결과 검증

### 예상 결과
✅ 승인자 토큰으로 문제 해결 작업이 실행되고 검증됨

---

# 정리

## 목표
Lab 3A의 기본 인프라는 유지하면서 Lab 3B 리소스를 제거합니다.

**삭제되는 리소스:**
- JWT 인증이 적용된 Gateway
- Lambda 인터셉터 함수
- Lambda 실행 역할

**유지되는 리소스:**
- AgentCore Runtime(Lab 3A에서 생성)
- Cognito User Pool 및 사용자
- OAuth2 자격 증명 공급자
- Parameter Store 항목

In [ ]:
from lab_helpers.config import AWS_REGION

# cleanup_lab_03b(region_name=AWS_REGION, verbose=True)

---

# 참고 자료
- Lab 3A: 문제 해결 에이전트 기초
- Cognito JWT 구성
- AgentCore Gateway 설명서
- Lambda 인터셉터 패턴